# Révision de croyances — JTMS, ATMS, résolution de conflits, Dung et QBF natifs

Cinquième volet de l'épic #1961 (Phase 4) : les **moteurs de révision de
croyances** du dépôt, distillés en un notebook exécutable. Tout ce qui suit
rejoue les moteurs **réels** — Python pur, zéro JVM, zéro LLM, zéro corpus —
et chaque cellule vérifie ses résultats contre
`belief_revision_examples.json` (généré par le même appel).

## Les cinq fichiers

| Module | Lignes | Rôle |
|---|---|---|
| `services/jtms/jtms_core.py` | 354 | JTMS — maintenance de vérité justificationniste, tri-états |
| `services/jtms/atms_core.py` | 242 | ATMS — environnements d'hypothèses, contrat d'étiquettes #2094 |
| `services/jtms/conflict_resolution.py` | 305 | 5 stratégies de résolution de conflits entre agents |
| `agents/core/logic/dung_native.py` | 308 | Sémantiques de Dung en Python natif (bornées #970) |
| `agents/core/logic/qbf_native.py` | 466 | QBF — logique quantifiée propositionnelle par énumération |

## Le fil conducteur : que veut dire « réviser » une croyance ?

Trois réponses cohabitent ici. Le **JTMS** révisionne *en place* : une croyance
a UNE valeur tri-état (vrai / faux / inconnu) qui se propage le long des
justifications. L'**ATMS** ne choisit pas : il conserve *tous* les mondes
d'hypothèses sous lesquels un nœud se dérive (ses étiquettes). La **résolution
de conflits** arbitre entre agents qui divergent. Dung et QBF tranchent
l'acceptabilité d'arguments par les extensions / quantificateurs.

In [1]:
import json
from pathlib import Path

EXAMPLES = json.loads(
    Path("docs/coursia_contrib/belief_revision_examples.json").read_text(encoding="utf-8")
)
assert EXAMPLES["asset"] == "belief_revision"
print("sections:", [k for k in EXAMPLES if k.endswith("_cases")])
print("phase:", EXAMPLES["phase"], "| issue:", EXAMPLES["issue"])

sections: ['jtms_cases', 'atms_cases', 'conflict_cases', 'dung_cases', 'qbf_cases']
phase: 4 | issue: 1961


## 1. JTMS — une vérité par croyance, propagée le long des justifications

Une `Justification` relie des prémisses `in_list` (toutes valides requises) et
`out_list` (toutes non-valides requises) à une conclusion. La croyance devient
`VALID` si une justification tire, sinon elle **retombe à `UNKNOWN`** — jamais
à `INVALID`, qui n'est atteint que par affectation directe. C'est le contrat
tri-états #2094 : `None` signifie « inconnu », jamais « invalide ».

In [2]:
from argumentation_analysis.services.jtms.jtms_core import JTMS

stored = {c["name"]: c for c in EXAMPLES["jtms_cases"]}

j = JTMS()
j.add_belief("A"); j.add_belief("B")
j.add_justification(["A"], [], "B")          # A justifie B
j.set_belief_validity("A", True)
print("A=vrai   :", j.beliefs["B"])
assert str(j.beliefs["B"]) == stored["propagation_and_retraction"]["b_after_a_true"]

j.set_belief_validity("A", False)
print("A=faux   :", j.beliefs["B"])
assert str(j.beliefs["B"]) == stored["propagation_and_retraction"]["b_after_a_false"]
print("-> rétracter la prémisse fait retomber B à UNKNOWN (pas INVALID)")

A=vrai   : B -> VALID
A=faux   : B -> UNKNOWN
-> rétracter la prémisse fait retomber B à UNKNOWN (pas INVALID)


### 1.1 Mode strict vs auto-création

`JTMS()` est **non-strict par défaut** : `add_justification` crée les croyances
manquantes. En mode strict, la même écriture lève `KeyError` — et
`set_belief_validity` sur un nom inconnu lève *dans tous les modes*.

In [3]:
s = stored["strict_mode_and_autocreate"]

j2 = JTMS(strict=True)
try:
    j2.set_belief_validity("X", True)
except KeyError as e:
    print("set inconnu (strict):", e)
    assert f"KeyError: {e}" == s["unknown_belief_set"]

try:
    j2.add_justification(["A"], [], "C")
    raise SystemExit("devrait lever")
except KeyError as e:
    print("add strict        :", e)
    assert f"KeyError: {e}" == s["strict_add_justification"]

nonstrict = JTMS(strict=False)
nonstrict.add_justification(["P"], [], "Q")   # P et Q auto-créés
print("auto-créés        :", sorted(nonstrict.beliefs))
assert sorted(nonstrict.beliefs) == s["nonstrict_autocreated_beliefs"]

set inconnu (strict): 'Unknown belief: X'
add strict        : 'Unknown belief: A'
auto-créés        : ['P', 'Q']


### 1.2 La `out_list` — justification non monotonique locale

`C` est justifié par `(in A, out B)` : la règle tire si A est valide **et que
B ne l'est pas**. Tant que B est inconnu la règle tire ; dès que B devient
valide, la règle se bloque et C retombe à `UNKNOWN`.

In [4]:
j3 = JTMS()
j3.add_justification(["A"], ["B"], "C")
j3.set_belief_validity("A", True)
print("B inconnu :", j3.beliefs["C"])
assert str(j3.beliefs["C"]) == stored["out_list_blocks_when_premise_valid"]["c_when_b_unknown"]

j3.set_belief_validity("B", True)
print("B valide  :", j3.beliefs["C"])
assert str(j3.beliefs["C"]) == stored["out_list_blocks_when_premise_valid"]["c_when_b_valid"]

B inconnu : C -> VALID
B valide  : C -> UNKNOWN


### 1.3 Cycles = non-monotonie verrouillée

Un cycle de justifications (A⇒B, B⇒A) forme une composante fortement connexe
(détectée via networkx) : ses membres sont marqués `non_monotonic` et
`compute_truth_statement` les **verrouille à `UNKNOWN`** — aucune vérité fixe
ne peut être attribuée à un support circulaire.

In [5]:
j4 = JTMS()
j4.add_justification(["A"], [], "B")
j4.add_justification(["B"], [], "A")
nonmono = sorted(n for n, b in j4.beliefs.items() if b.non_monotonic)
print("membres du cycle marqués non_monotonic :", nonmono)
assert nonmono == stored["scc_cycle_marks_non_monotonic_and_locks_unknown"]["cycle_members"]
print("A.valid après calcul :", j4.beliefs["A"].valid, "(verrouillé à None)")

membres du cycle marqués non_monotonic : ['A', 'B']
A.valid après calcul : None (verrouillé à None)


### 1.4 `explain_belief` — verdicts tri-états par justification

Pour chaque justification : `Valid` (tire), `Invalid` (une prémisse in est
fausse **ou** une prémise out est vraie — réfutation), `Unknown` (indéterminé).
Le cas out ci-dessous montre les trois verdicts sur la même règle.

In [6]:
j5 = JTMS()
j5.add_justification(["A"], ["B"], "C")
j5.set_belief_validity("A", True)

def verdict():
    return j5.explain_belief("C").strip().splitlines()[-1]

v1 = verdict(); print("B inconnu ->", v1)
assert v1 == stored["explain_belief_tri_state_verdicts"]["unknown_premise"]

j5.set_belief_validity("B", True)
v2 = verdict(); print("B valide  ->", v2)     # out vraie = réfutation
assert v2 == stored["explain_belief_tri_state_verdicts"]["refuted_by_out_premise"]

j5.set_belief_validity("B", False)
v3 = verdict(); print("B faux    ->", v3)
assert v3 == stored["explain_belief_tri_state_verdicts"]["valid_when_out_falsy"]

B inconnu ->   Result: Valid
B valide  ->   Result: Invalid
B faux    ->   Result: Valid


### 1.5 Tracer une rétraction en cascade + démolition propre

`enable_tracing()` enregistre chaque rétraction : déclencheur, motif, et la
cascade des conclusions emportées. `remove_belief` (#2094) détache la règle
**dans les deux sens** — les prémisses survivantes ne gardent aucune référence
vers la conclusion disparue.

In [7]:
j6 = JTMS()
j6.enable_tracing()
j6.add_justification(["A"], [], "B")
j6.add_justification(["B"], [], "C")
j6.set_belief_validity("A", True)
j6.set_belief_validity("A", None)             # rétractation
trace = j6.get_retraction_chain()
print(json.dumps(trace, ensure_ascii=False, indent=1))
assert trace == stored["retraction_trace_cascade"]["trace"]

j7 = JTMS()
j7.add_justification(["A"], [], "B")
j7.set_belief_validity("A", True)
j7.remove_belief("A")
print("après remove_belief(A) :", sorted(j7.beliefs), "| justifs de B :", len(j7.beliefs["B"].justifications))
s7 = stored["remove_belief_tears_down_justifications"]
assert sorted(j7.beliefs) == s7["beliefs_after"]
assert len(j7.beliefs["B"].justifications) == s7["b_justifications_after"] == 0

[
 {
  "trigger": "A",
  "retracted": [
   "A"
  ],
  "cascaded": [
   "C",
   "B"
  ],
  "reason": "directly set to None"
 }
]
après remove_belief(A) : ['B'] | justifs de B : 0


## 2. ATMS — tous les mondes d'hypothèses à la fois

L'ATMS ne propage pas une vérité : il étiquette chaque nœud par l'ensemble des
**environnements** (ensembles d'hypothèses) sous lesquels il se dérive. Une
hypothèse porte sa propre étiquette `{elle-même}` ; une justification fait le
**produit** des étiquettes de ses entrées et fusionne les environnements.

**Contrat d'étiquettes #2094** (arbitrage R994 — le notebook enseigne l'API
réelle) : les étiquettes ne poussent **qu'à l'insertion** d'une justification
et ne sont retirées **que par `invalidate_environment`**. Il n'existe ni API
de retrait d'hypothèse, ni re-propagation : un environnement invalidé reste
absent jusqu'à ce qu'une *nouvelle* justification le redérive.

In [8]:
from argumentation_analysis.services.jtms.atms_core import ATMS, CONTRADICTION_SYMBOL

stored = {c["name"]: c for c in EXAMPLES["atms_cases"]}

a = ATMS()
a.add_assumption("p"); a.add_assumption("q")
a.add_node("r")
a.add_justification(["p", "q"], [], "r")
envs = lambda n: sorted(sorted(e) for e in a.get_environments(n))
print("p :", envs("p"))
print("r :", envs("r"), "<- produit {p} × {q}")
s0 = stored["assumption_label_and_product_environment"]
assert envs("p") == [sorted(e) for e in [frozenset({"p"})]] and envs("p") == s0["p_envs"]
assert envs("r") == s0["r_envs"]

p : [['p']]
r : [['p', 'q']] <- produit {p} × {q}


### 2.1 `invalidate_environment` ne redérive jamais

Invalider `{x}` retire `{x}` et tous ses sur-ensembles de **chaque** nœud — y
compris de l'hypothèse `x` elle-même. Aucune mécanique ne reconstruit les
étiquettes : la dérivation `[x] → t` existe toujours mais ne re-tire pas.

In [9]:
s1 = stored["invalidation_strips_and_never_repropagates"]
a2 = ATMS()
a2.add_assumption("x"); a2.add_node("t")
a2.add_justification(["x"], [], "t")
print("t avant :", sorted(sorted(e) for e in a2.get_environments("t")))
a2.invalidate_environment(frozenset({"x"}))
print("t après :", [sorted(e) for e in a2.get_environments("t")])
print("x après :", [sorted(e) for e in a2.get_environments("x")])
assert [sorted(e) for e in a2.get_environments("t")] == s1["t_envs_after_invalidation"] == []
assert [sorted(e) for e in a2.get_environments("x")] == s1["assumption_x_envs_after"] == []
# La justification est TOUJOURS là — rien ne re-tire :
assert len(a2.nodes["t"].justifications) == 1

t avant : [['x']]
t après : []
x après : []


### 2.2 Le nogood n'est pas durable — et l'invalidation tire deux fois

Conclure vers `⊥` enregistre l'environnement contradictoire **puis invoque
immédiatement `invalidate_environment`** — qui nettoie `⊥` lui-même : le
nogood n'est pas mémorisé, `is_consistent` redevient vrai. Piège mesuré : le
produit cartésien sur une étiquette non minimale engendre **deux** environnements
(`{m,n}` puis `{m}`) — donc **deux** invalidations, et la dernière (`{m}`)
strippe `{m}` partout, y compris l'étiquette propre de l'hypothèse `m`.

In [10]:
s2 = stored["contradiction_nogood_not_durable"]
a3 = ATMS()
a3.add_assumption("m"); a3.add_assumption("n")
a3.add_justification(["m"], [], "n")              # n : {{n}, {m}}  (non minimal !)
print("consistent({m,n}) avant :", a3.is_consistent(frozenset({"m", "n"})))
a3.add_justification(["m", "n"], [], CONTRADICTION_SYMBOL)
print("consistent({m,n}) après :", a3.is_consistent(frozenset({"m", "n"})))
print("⊥ :", [sorted(e) for e in a3.get_environments(CONTRADICTION_SYMBOL)])
print("n :", [sorted(e) for e in a3.get_environments("n")], "| m :", [sorted(e) for e in a3.get_environments("m")])
assert a3.is_consistent(frozenset({"m", "n"})) is s2["consistent_mn_after_nogood"] is True
assert [sorted(e) for e in a3.get_environments("n")] == s2["n_envs_after"]
assert [sorted(e) for e in a3.get_environments("m")] == s2["m_envs_after"] == []

consistent({m,n}) avant : True
consistent({m,n}) après : True
⊥ : []
n : [['n']] | m : []


### 2.3 Blocage par nœud out, nœud inconnu, explication

Une entrée `out` bloque tout environnement dont elle est un **sous-ensemble** :
`w` étant dérivable sous `{u}`, la conclusion `(in v, out w)` ne garde que les
environnements évitant `u`.

In [11]:
a4 = ATMS()
a4.add_assumption("s"); a4.add_assumption("u")
a4.add_node("v"); a4.add_node("w"); a4.add_node("blocked")
a4.add_justification(["s"], [], "v")
a4.add_justification(["u"], [], "w")
a4.add_justification(["v"], ["w"], "blocked")
blocked = [sorted(e) for e in a4.get_environments("blocked")]
print("blocked :", blocked, "(pas de {s,u} : bloqué par w)")
assert blocked == stored["out_node_blocks_superset_environments"]["blocked_envs"]

try:
    a4.get_environments("zz")
except KeyError as e:
    print("nœud inconnu :", e)
    assert f"KeyError: {e}" == stored["unknown_node_raises"]["result"]

a5 = ATMS(); a5.add_assumption("d"); a5.add_node("e")
a5.add_justification(["d"], [], "e")
print(json.dumps(a5.explain_node("e"), ensure_ascii=False))
assert a5.explain_node("e") == stored["explain_node_shape"]["explain"]

blocked : [['s']] (pas de {s,u} : bloqué par w)
nœud inconnu : "Node 'zz' not found in ATMS."
{"name": "e", "is_assumption": false, "environments": [["d"]], "justifications": [{"in_nodes": ["d"], "out_nodes": []}]}


## 3. Résolution de conflits — cinq stratégies, un même contrat

`ConflictResolver.resolve(conflict, strategy=...)` reçoit des croyances
divergentes d'agents et rend `{resolved, chosen_agent, reasoning, strategy_used, ...}`.
Cinq stratégies : `confidence_based`, `evidence_based`, `consensus`,
`agent_expertise`, `temporal`. Une stratégie **inconnue lève `ValueError`**,
et un conflit sans `beliefs` aussi : les deux viennent du code appelant.
Jusqu'à #2344, une stratégie inconnue retombait silencieusement sur
`confidence_based`. Une croyance **sans confiance** n'est pas comptée 0.0 :
la comparaison est rendue non résolue et nomme l'agent (`missing_confidence`).

In [12]:
from argumentation_analysis.services.jtms.conflict_resolution import ConflictResolver

stored = {c["name"]: c for c in EXAMPLES["conflict_cases"]}
r = ConflictResolver()

conflict = {
    "belief_name": "hypothesis_X",
    "beliefs": {
        "agent_1": {"belief_name": "hypothesis_X", "confidence": 0.8, "valid": True},
        "agent_2": {"belief_name": "hypothesis_X", "confidence": 0.3, "valid": False},
    },
    "context": {"type": "hypothesis"},
}
res = r.resolve(conflict, strategy="confidence_based")
print(res["chosen_agent"], "|", res["reasoning"])
assert res["chosen_agent"] == stored["confidence_based"]["chosen_agent"]

s_unknown = stored["unknown_strategy_raises"]
try:
    r.resolve(conflict, strategy="made_up_strategy")
    raise AssertionError("une stratégie inconnue doit lever")
except ValueError as e:
    print("stratégie inconnue ->", type(e).__name__, "|", e)
    assert type(e).__name__ == s_unknown["raises"]
    assert s_unknown["message_contains"] in str(e)

ev = {
    "belief_name": "claim_Y",
    "beliefs": {
        "agent_1": {"belief_name": "claim_Y", "confidence": 0.5, "justification_count": 4},
        "agent_2": {"belief_name": "claim_Y", "confidence": 0.9, "justification_count": 1},
    },
}
res_ev = r.resolve(ev, strategy="evidence_based")   # score = count × confiance
print("evidence :", res_ev["chosen_agent"], "|", res_ev["reasoning"])
assert res_ev["chosen_agent"] == stored["evidence_based_score_beats_confidence"]["chosen_agent"]
print("-> 4×0.5 = 2.0 bat 1×0.9 : le score, pas la confiance nue, décide")

agent_1 | Highest confidence: 0.80 by agent_1
stratégie inconnue -> ValueError | Unknown conflict resolution strategy 'made_up_strategy'; known: confidence_based, evidence_based, consensus, agent_expertise, temporal
evidence : agent_1 | Best evidence score: 2.00 by agent_1
-> 4×0.5 = 2.0 bat 1×0.9 : le score, pas la confiance nue, décide


### 3.1 Consensus (quorum 3+), expertise, temporel

Le consensus exige **au moins 3 agents** et tranche à la majorité ; une
égalité rend `resolved=False` (« Consensus tie »). L'expertise mappe le type
de contexte à un agent expert (sherlock pour `evidence`…) par **sous-chaîne
insensible à la casse** sur le nom d'agent. Le temporel compare les horodatages
**comme chaînes** (ISO court donc correct).

Quand la stratégie ne s'applique pas (aucun agent expert, domaine inconnu,
aucun horodatage), le résolveur se replie sur la confiance **et le dit** :
`strategy_used` nomme ce qui a décidé (`confidence_based`) et `fallback_from`
ce qui était demandé. Avant #2344, `strategy_used` gardait le nom demandé, et
un domaine inconnu désignait sherlock comme expert.

In [13]:
s3 = stored["consensus_quorum_and_tie"]
two = r.resolve(conflict, strategy="consensus")
print("2 agents  :", two["resolved"], "|", two["reasoning"])
assert two["reasoning"] == s3["two_agents_reasoning"]

three = r.resolve({
    "belief_name": "hyp_Z",
    "beliefs": {
        "a1": {"belief_name": "hyp_Z", "confidence": 0.9, "valid": True},
        "a2": {"belief_name": "hyp_Z", "confidence": 0.5, "valid": True},
        "a3": {"belief_name": "hyp_Z", "confidence": 0.1, "valid": False},
    },
}, strategy="consensus")
print("3 agents  :", three["reasoning"])
assert three["reasoning"] == s3["three_agents_reasoning"]

tie = r.resolve({
    "belief_name": "hyp_T",
    "beliefs": {f"a{i}": {"belief_name": "hyp_T", "valid": i % 2 == 0} for i in range(1, 5)},
}, strategy="consensus")
print("égalité   :", tie["resolved"], "|", tie["reasoning"])
assert tie["resolved"] is s3["tie_resolved"] is False

s4 = stored["agent_expertise_substring_and_fallback"]
exp = r.resolve({
    "belief_name": "ev_W",
    "beliefs": {
        "watson_agent": {"belief_name": "ev_W", "confidence": 0.2},
        "sherlock_agent": {"belief_name": "ev_W", "confidence": 0.9},
    },
    "context": {"type": "evidence"},
}, strategy="agent_expertise")
print("expert    :", exp["chosen_agent"], "|", exp["reasoning"])
assert exp["chosen_agent"] == s4["expert_chosen"]

noexp = r.resolve({
    "belief_name": "ev_N",
    "beliefs": {
        "agent_x": {"belief_name": "ev_N", "confidence": 0.4},
        "agent_y": {"belief_name": "ev_N", "confidence": 0.7},
    },
    "context": {"type": "evidence"},
}, strategy="agent_expertise")
print("sans expert :", noexp["chosen_agent"], "|", noexp["strategy_used"],
      "(repli depuis", noexp["fallback_from"] + ")")
print("   ", noexp["reasoning"])
assert noexp["chosen_agent"] == s4["no_expert_fallback_agent"]
assert noexp["reasoning"] == s4["no_expert_fallback_reasoning"]
assert noexp["fallback_from"] == s4["no_expert_fallback_from"]

s5 = stored["temporal_iso_string_comparison"]
t = r.resolve({
    "belief_name": "fact_T",
    "beliefs": {
        "old": {"belief_name": "fact_T", "timestamp": "2026-01-01T10:00:00"},
        "new": {"belief_name": "fact_T", "timestamp": "2026-06-01T10:00:00"},
    },
}, strategy="temporal")
print("temporel  :", t["chosen_agent"])
assert t["chosen_agent"] == s5["latest_chosen"]

t_none = r.resolve({
    "belief_name": "fact_U",
    "beliefs": {
        "a": {"belief_name": "fact_U", "confidence": 0.3},
        "b": {"belief_name": "fact_U", "confidence": 0.6},
    },
}, strategy="temporal")
print("sans horodatage :", t_none["chosen_agent"], "|", t_none["strategy_used"],
      "(repli depuis", t_none["fallback_from"] + ")")
assert t_none["chosen_agent"] == s5["no_timestamp_fallback_agent"]
assert t_none["fallback_from"] == s5["no_timestamp_fallback_from"]

stats = r.get_stats()
print("stats de CETTE session :", json.dumps(stats))
hist = stored["history_stats"]["stats"]
print("stats stockées          :", json.dumps(hist))
assert stats["total_conflicts"] == stats["resolved"] + stats["unresolved"]
assert stats == hist
print("-> les replis sont comptés sous confidence_based (3) ; la stratégie inconnue,"
      " qui lève, n'entre pas dans l'historique")

2 agents  : False | Consensus requires 3+ agents
3 agents  : Consensus: 2 for vs 1 against
égalité   : False | Consensus tie
expert    : sherlock_agent | Expert sherlock for domain evidence
sans expert : agent_y | confidence_based (repli depuis agent_expertise)
    agent_expertise not applicable (no sherlock agent among the beliefs); fell back to confidence. Highest confidence: 0.70 by agent_y
temporel  : new
sans horodatage : b | confidence_based (repli depuis temporal)
stats de CETTE session : {"total_conflicts": 9, "resolved": 7, "unresolved": 2, "by_strategy": {"confidence_based": 3, "evidence_based": 1, "consensus": 3, "agent_expertise": 1, "temporal": 1}}
stats stockées          : {"total_conflicts": 9, "resolved": 7, "unresolved": 2, "by_strategy": {"confidence_based": 3, "evidence_based": 1, "consensus": 3, "agent_expertise": 1, "temporal": 1}}
-> les replis sont comptés sous confidence_based (3) ; la stratégie inconnue, qui lève, n'entre pas dans l'historique


## 4. Dung natif — sémantiques sous borne d'énumération

`DungFramework` calcule grounded / preferred / stable / complete en Python
pur. **Borne #970** : `_MAX_ENUM_ARGS = 15` — au-delà, l'énumération
(admissible, stable…) lève `RuntimeError` ; le **grounded** reste disponible
(itération de la fonction caractéristique, pas d'énumération).

In [14]:
from argumentation_analysis.agents.core.logic.dung_native import DungFramework

stored = {c["name"]: c for c in EXAMPLES["dung_cases"]}

tri = DungFramework.triangle()          # a→b→c→a
fmt = lambda fw, f: [sorted(e) for e in getattr(fw, f)()]
print("triangle  grounded :", fmt(tri, "grounded_extension"))
print("triangle preferred :", fmt(tri, "preferred_extensions"))
print("triangle stable    :", fmt(tri, "stable_extensions"))
s0 = stored["triangle_odd_cycle_nothing_accepted"]
assert sorted(tri.grounded_extension()) == s0["grounded"] == []
assert fmt(tri, "preferred_extensions") == s0["preferred"] == [[]]   # ∅ est le seul admissible
assert fmt(tri, "stable_extensions") == s0["stable"] == []
print("-> cycle impair : aucun singleton ne se défend ; ∅ est admissible mais pas stable")

triangle  grounded : []
triangle preferred : [[]]
triangle stable    : []
-> cycle impair : aucun singleton ne se défend ; ∅ est admissible mais pas stable


In [15]:
rei = DungFramework.reinstatement()      # a→b→c
print("réinstauration grounded :", sorted(rei.grounded_extension()))
print("-> a attaque b, qui attaquait c : c est réinstallé par a")
s1 = stored["reinstatement_a_defends_c"]
assert sorted(rei.grounded_extension()) == s1["grounded"]
assert fmt(rei, "preferred_extensions") == s1["preferred"]
assert fmt(rei, "stable_extensions") == s1["stable"]
print("F(∅)  =", sorted(rei.characteristic_function(frozenset())))
print("F({a}) =", sorted(rei.characteristic_function(frozenset({"a"}))))
s5 = stored["characteristic_function"]
assert sorted(rei.characteristic_function(frozenset())) == s5["reinstatement_F_empty"]
assert sorted(rei.characteristic_function(frozenset({"a"}))) == s5["reinstatement_F_a"]

réinstauration grounded : ['a', 'c']
-> a attaque b, qui attaquait c : c est réinstallé par a
F(∅)  = ['a']
F({a}) = ['a', 'c']


### 4.1 `mutual_destruction` — la variante à 4 arguments du dépôt

Attention : la méthode ne construit **pas** le diamant classique à deux
extensions stables. Les arêtes mesurées sont `quaker→hawk`,
`republican→pacifist`, `hawk↔pacifist` : quaker et republican sont des
prémisses non attaquées et hawk/pacifist sont rejetés sous **toutes** les
sémantiques. Anciennement nommée `nixon_diamond`, renommée `mutual_destruction`
(#2252) : le nom encode ce que les arêtes construisent. Le diamant canonique
de Nixon (deux extensions stables) vit dans `abs_arg_dung` sous le nom
`nixon_diamond`.


In [16]:
nix = DungFramework.mutual_destruction()
print(json.dumps(nix.get_all_extensions(), ensure_ascii=False, indent=1))
print("hawk    :", nix.get_argument_status("hawk"))
print("pacifist:", nix.get_argument_status("pacifist"))
s2 = stored["nixon_diamond_measured"]
assert nix.get_all_extensions() == s2["all_extensions"]
assert nix.get_argument_status("hawk") == s2["status_hawk"]
assert nix.get_argument_status("pacifist") == s2["status_pacifist"]

tri_props = DungFramework.triangle().framework_properties()
self_atk = DungFramework(); self_atk.add_attack("s", "s")
print("triangle props :", tri_props)
print("self-attacking :", self_atk.framework_properties())
s3 = stored["framework_properties"]
assert tri_props == s3["triangle"]
assert self_atk.framework_properties() == s3["self_attacking"]

{
 "grounded": [
  [
   "quaker",
   "republican"
  ]
 ],
 "preferred": [
  [
   "quaker",
   "republican"
  ]
 ],
 "stable": [
  [
   "quaker",
   "republican"
  ]
 ],
 "complete": [
  [
   "quaker",
   "republican"
  ]
 ],
 "admissible": [
  [],
  [
   "quaker"
  ],
  [
   "republican"
  ],
  [
   "quaker",
   "republican"
  ]
 ]
}
hawk    : {'in_grounded': False, 'credulously_accepted': False, 'skeptically_accepted': False, 'in_stable': False}
pacifist: {'in_grounded': False, 'credulously_accepted': False, 'skeptically_accepted': False, 'in_stable': False}
triangle props : {'num_arguments': 3, 'num_attacks': 3, 'has_cycles': True, 'self_attacking': [], 'unattacked': []}
self-attacking : {'num_arguments': 1, 'num_attacks': 1, 'has_cycles': True, 'self_attacking': ['s'], 'unattacked': []}


In [17]:
big = DungFramework()
for i in range(16):
    big.add_argument(f"x{i}")
for i in range(15):
    big.add_attack(f"x{i}", f"x{i+1}")
grounded_big = sorted(big.grounded_extension())
print("grounded (16 args) :", grounded_big, "<- les indices pairs, sans énumération")
try:
    big.stable_extensions()
    raise SystemExit("devrait lever")
except RuntimeError as err:
    cap_msg = str(err)
    print("stable :", cap_msg)
s4 = stored["enumeration_cap_970"]
assert grounded_big == s4["grounded_still_works"]
assert cap_msg == s4["stable_raises"]
assert "#970" in cap_msg

grounded (16 args) : ['x0', 'x10', 'x12', 'x14', 'x2', 'x4', 'x6', 'x8'] <- les indices pairs, sans énumération
stable : Framework has 16 arguments, exceeding enumeration limit (15). Stable extensions computation is unavailable. (#970)


## 5. QBF natif — quantificateurs par énumération naïve

`check_qbf` construit les quantificateurs imbriqués et énumère l'espace de
recherche (2^n borné à n≤15). Les statistiques rendues nomment le trio
`handler: qbf_native`, `reasoner: naive_enumeration`, `search_space: 2^n`.

In [18]:
from argumentation_analysis.agents.core.logic import qbf_native as qbf

stored = {c["name"]: c for c in EXAMPLES["qbf_cases"]}

print(qbf.example_simple_validity()["message"])
print(qbf.example_simple_satisfiability()["message"])
print(qbf.example_mixed_quantifiers()["message"])
assert qbf.example_simple_validity() == stored["tautology"]["analysis"]
assert qbf.example_simple_satisfiability() == stored["contradiction"]["analysis"]
assert qbf.example_mixed_quantifiers() == stored["mixed_quantifiers_valid"]["analysis"]
print("stats:", qbf.example_mixed_quantifiers()["statistics"])

QBF VALID: x | !x
QBF INVALID: x & !x
QBF VALID: x => y
stats: {'quantifier_count': 2, 'variable_count': 2, 'search_space': 4, 'handler': 'qbf_native', 'reasoner': 'naive_enumeration'}


### 5.1 Le parseur : précédences, pas de parenthèses

Le mini-parseur ne connaît **pas les parenthèses** — elles deviennent partie
du nom de variable. Précérences : `!` > `&` > `|` > `=>` ; `=>` est
associatif **à droite**, `|` et `&` se replient **à gauche**. Et
`Var.evaluate` rend **False** pour une variable non affectée (pas d'erreur).

In [19]:
print(repr(qbf.parse_formula("a & (b | c)")))
print("-> '(b' et 'c)' sont des NOMS de variables : le | n'est pas dans la parenthèse")
assert repr(qbf.parse_formula("a & (b | c)")) == stored["parser_no_parentheses_support"]["repr_of_a_and_paren_expr"]

print(repr(qbf.parse_formula("a => b => c")), "<- droite")
assert repr(qbf.parse_formula("a => b => c")) == stored["parser_implication_right_associative"]["repr"]
print(repr(qbf.parse_formula("a | b | c")), "<- gauche")
assert repr(qbf.parse_formula("a | b | c")) == stored["parser_or_left_fold"]["repr"]

print("Var('zz').evaluate({}) =", qbf.Var("zz").evaluate({}))
assert qbf.Var("zz").evaluate({}) is stored["unassigned_variable_defaults_false"]["eval_empty_assignment"] is False

((a & (b) | c))
-> '(b' et 'c)' sont des NOMS de variables : le | n'est pas dans la parenthèse
(a => (b => c)) <- droite
((a | b) | c) <- gauche
Var('zz').evaluate({}) = False


### 5.2 Acceptabilité crédule et sceptique via Dung

`credulous_acceptance_qbf` cherche un **témoin** (une extension admissible
contenant la cible) ; `skeptical_acceptance_qbf` exige la cible dans
**toutes** les extensions preferred (pont vers Dung). Au-delà de 15 arguments
l'acceptabilité rend `accepted: None` (indéterminé), pas d'erreur.

In [20]:
witness = qbf.example_argumentation_acceptance()
print(json.dumps(witness, ensure_ascii=False))
assert witness == stored["credulous_witness"]["result"]

sk = qbf.skeptical_acceptance_qbf(
    arguments=["a", "b", "c"], attacks=[["a", "b"], ["b", "a"]], target="a",
)
print(json.dumps(sk, ensure_ascii=False, indent=1))
assert sk == stored["skeptical_via_preferred"]["result"]
print("-> {a,c} et {b,c} sont preferred : 'a' n'est pas sceptiquement accepté")

absent = qbf.skeptical_acceptance_qbf(["a"], [], "nope")
print("cible absente :", absent["reason"])
assert absent == stored["absent_target"]["result"]

big_args = [f"y{i}" for i in range(16)]
big = qbf.credulous_acceptance_qbf(big_args, [], "y0")
print("16 args :", big["accepted"], "|", big["reason"])
assert big == stored["too_large_returns_none"]["result"]

{"target": "a", "accepted": true, "witness_extension": ["a"], "reason": "Found admissible extension containing a", "method": "credulous_qbf"}
{
 "target": "a",
 "accepted": false,
 "preferred_extensions": [
  [
   "a",
   "c"
  ],
  [
   "b",
   "c"
  ]
 ],
 "reason": "a is NOT in all preferred extensions",
 "method": "skeptical_qbf"
}
-> {a,c} et {b,c} sont preferred : 'a' n'est pas sceptiquement accepté
cible absente : Argument 'nope' not in framework
16 args : None | Framework too large (16 args) for naive enumeration


## Récapitulatif — ce que le dépôt enseigne vraiment

| Système | Réponse à « réviser » | Piège mesuré principal |
|---|---|---|
| JTMS | une vérité tri-état propagée en place | rétracter une prémisse retombe à UNKNOWN, jamais INVALID |
| ATMS | toutes les environnements dérivatives à la fois | nogood non durable ; produit non minimal ⇒ double invalidation |
| Conflits | arbitre entre agents | l'expertise matche par sous-chaîne du nom d'agent ; le repli confiance est nommé (`fallback_from`) depuis #2344, qui a aussi rendu fatale la stratégie inconnue |
| Dung | acceptabilité par extensions | borne d'énumération #970 (grounded survit) |
| QBF | quantificateurs par énumération | parseur sans parenthèses ; Var non affectée = False |

Corpus-free · zéro LLM · zéro JVM — chaque sortie de cellule est le résultat
réel d'une exécution, vérifié contre `belief_revision_examples.json`.